In [109]:
import numpy as np
import pandas as pd

In [110]:
df = pd.read_csv('diabetes.csv')

In [111]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1


In [112]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.495990
BloodPressure,0.174469
SkinThickness,0.295138
Insulin,0.377081
BMI,0.315577
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [113]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [114]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [115]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Input

# How to select appropriate :-
# 1. Optimizer Value
# 2. Number of Nodes in a Layer
# 3. Number of Layers
# 4. All in All one Model

In [117]:
import keras_tuner as kt

In [42]:
# 1.

# hp = hyperparamter
def build_model(hp):
  model = Sequential()

  model.add(Input(shape=(8,)))
  model.add(Dense(32,activation='relu'))
  model.add(Dense(1,activation='sigmoid'))

  optimizer = hp.Choice('optimizer',values = ['adam','sgd','rmsprop','adadelta'])

  # Now it will create 4 different model with different optimizers
  model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])

  return model

In [43]:
# Create a RandomSearch tuner to try 5 different hyperparameter combinations and select the best based on validation accuracy
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5,project_name='optimizer_test',overwrite=True)

In [44]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))


Trial 4 Complete [00h 00m 02s]
val_accuracy: 0.6688311696052551

Best val_accuracy So Far: 0.701298713684082
Total elapsed time: 00h 00m 10s


In [45]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'sgd'}

In [46]:
model = tuner.get_best_models(num_models=1)[0]

In [59]:
# 2.

# hp = hyperparamter
def build_model(hp):
  model = Sequential()

  # we can also remove step size
  units = hp.Int('units',min_value = 8,max_value = 128,step=8)

  model.add(Input(shape=(8,)))
  model.add(Dense(units=units,activation='relu'))
  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer='sgd',loss='binary_crossentropy',metrics=['accuracy'])

  return model

In [69]:
# Create a RandomSearch tuner to try 5 different hyperparameter combinations and select the best based on validation accuracy
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5,project_name='optimizer_test',overwrite=True)

In [70]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 02s]
val_accuracy: 0.7142857313156128

Best val_accuracy So Far: 0.7597402334213257
Total elapsed time: 00h 00m 11s


In [71]:
tuner.get_best_hyperparameters()[0].values

{'units': 96}

In [72]:
model = tuner.get_best_models(num_models=1)[0]

In [83]:
# 3.

# hp = hyperparamter
def build_model(hp):
  model = Sequential()

  model.add(Input(shape=(8,)))
  model.add(Dense(96,activation='relu'))

  for i in range(hp.Int('num_layers',min_value=1,max_value=10)):
    model.add(Dense(96,activation='relu'))

  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer='sgd',loss='binary_crossentropy',metrics=['accuracy'])

  return model

In [85]:
# Create a RandomSearch tuner to try 5 different hyperparameter combinations and select the best based on validation accuracy
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=3,project_name='optimizer_test',overwrite=True)

In [86]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 02s]
val_accuracy: 0.7402597665786743

Best val_accuracy So Far: 0.7402597665786743
Total elapsed time: 00h 00m 07s


In [87]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 7}

In [89]:
model = tuner.get_best_models(num_models=1)[0]

In [ ]:
# 4.

# hp = hyperparamter
def build_model(hp):
  model = Sequential()

  counter = 0
    

  for i in range(hp.Int('num_layers',min_value=1,max_value=10)):

    # Tune units and activation separately for each hidden layer
    # Unique HP names (units_0, units_1, ...) make them independent hyperparameters
    units = hp.Int( f'units_{i}',min_value = 8, max_value = 128, step = 8)
    activation = hp.Choice(f'activation_{i}',values=['relu','tanh','sigmoid'])
    
    if counter == 0:
      model.add(Input(shape=(8,)))
      model.add(Dense(units=units,activation=activation))
      

    else:  
      model.add(Dense(units=units,activation=activation))
      
    # Tune dropout separately for each hidden layer using unique HP names
    model.add(Dropout(hp.Choice(f'dropout_{i}',values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
    counter += 1

  model.add(Dense(1,activation='sigmoid'))

  # Optimizer is shared by the entire model, so keep it outside the loop
  optimizer = hp.Choice('optimizer',values=['sgd','rmsprop','adam','adadelta'])
  model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])
  
  return model

In [140]:
# Create a RandomSearch tuner to try 5 different hyperparameter combinations and select the best based on validation accuracy
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=3,project_name='optimizer_test',overwrite=True)

In [141]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 03s]
val_accuracy: 0.7467532753944397

Best val_accuracy So Far: 0.7467532753944397
Total elapsed time: 00h 00m 10s


In [142]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 2,
 'units_0': 104,
 'activation_0': 'tanh',
 'optimizer': 'adam',
 'units_1': 112,
 'activation_1': 'tanh',
 'units_2': 40,
 'activation_2': 'tanh',
 'units_3': 112,
 'activation_3': 'tanh',
 'units_4': 80,
 'activation_4': 'sigmoid'}

In [143]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [144]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 104)            │           936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 112)            │        11,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           113 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,809 (50.04 KB)

 Trainable params: 12,809 (50.04 KB)

 Non-trainable params: 0 (0.00 B)

In [145]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7997 - loss: 0.4465 - val_accuracy: 0.7273 - val_loss: 0.5265
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8094 - loss: 0.4185 - val_accuracy: 0.7338 - val_loss: 0.5255
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8225 - loss: 0.4064 - val_accuracy: 0.7857 - val_loss: 0.5128
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8078 - loss: 0.4203 - val_accuracy: 0.7922 - val_loss: 0.5384
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8094 - loss: 0.4102 - val_accuracy: 0.7857 - val_loss: 0.5287
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8404 - loss: 0.3901 - val_accuracy: 0.7857 - val_loss: 0.5059
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8388 - loss: 0.3886 - val_accuracy: 0.8117 - val_loss: 0.4976
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8306 - loss: 0.3801 - val_accuracy: 0.74